# SmolVLM 256M experiment record
Original source variant, not a separate result.

Original: `V3_EdgeVLM_256M_Phase2_MediaPipe_Enhanced.ipynb`. Numeric/text outputs are retained where present. Embedded visual outputs and image-path previews are removed for publication. Execution counts are not fabricated. See README.md and the source manifest.

This is a historical server notebook, not a portable CLI. Review paths, dependencies, adapter initialization and optional export cells before executing. No dataset or weights are included.


# EdgeVLM-Drive — SmolVLM-256M Phase 2 with selected techniques

This is a new, isolated experiment. It does not overwrite the earlier 256M model.

Applied techniques:

1. Phase 2 subject-disjoint train/validation/test splits with training-path deduplication.
2. Production-aligned enhancement and full-frame preprocessing imported from `langgraph_pipeline.py`.
3. Consistent dual-crop input: full available frame plus a face crop guided by MediaPipe landmarks when detected; the same crop function is used for training, validation, and test inference.
4. Cached MediaPipe face landmarks and blendshape cues, extracted without using ground-truth labels.
5. LoRA on both vision-encoder and text-decoder attention projections.
6. WeightedRandomSampler for class-balanced training exposure.
7. Up to eight epochs with early stopping selected by generated validation Macro-F1.
8. Checkpoints, resume support, terminal-friendly ETA, and resumable full-test evaluation.

The default starts from the previous good 256M adapter, merges it into the base, and
then trains a fresh vision+text LoRA adapter. For a controlled comparison against the
500M experiment, set `START_FROM_PREVIOUS_ADAPTER=False` so both models start from
their untouched pretrained bases.


## 1. Configuration and production preprocessing


In [ ]:
import os
import sys
import importlib.util
from pathlib import Path

# Keep this experiment on one GPU. This avoids the DataParallel stall observed earlier.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

HOME = Path("/home/jovyan")
SPLIT_DIR = HOME / "phase2-tain-val-test-splits"
TRAIN_CSV = SPLIT_DIR / "final_train.csv"
VAL_CSV = SPLIT_DIR / "final_val.csv"
TEST_CSV = SPLIT_DIR / "final_test.csv"

PROJECT_ROOT = HOME / "EdgeVLM_finetune vlm256"
OUTPUT_ROOT = PROJECT_ROOT / "phase2_256m_mediapipe_targetaug_sqrt_150_per_class_patience3"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
LAST_ADAPTER_DIR = OUTPUT_ROOT / "last_adapter"
BEST_ADAPTER_DIR = OUTPUT_ROOT / "best_by_macro_f1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
MODEL_CACHE_CANDIDATES = [
    PROJECT_ROOT / "model_cache",
    HOME / "EdgeVLM_finetune" / "model_cache",
]
MODEL_CACHE = next((p for p in MODEL_CACHE_CANDIDATES if p.exists()), MODEL_CACHE_CANDIDATES[0])
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

PREVIOUS_ADAPTER_CANDIDATES = [
    PROJECT_ROOT / "smolvlm_256m_lora_augmented_classweighted",
    HOME / "EdgeVLM_finetune_vlm256" / "smolvlm_256m_lora_augmented_classweighted",
    HOME / "EdgeVLM_finetune" / "smolvlm_256m_lora_augmented_classweighted",
]
PREVIOUS_ADAPTER_DIR = next(
    (p for p in PREVIOUS_ADAPTER_CANDIDATES if (p / "adapter_model.safetensors").is_file()),
    None,
)

START_FROM_PREVIOUS_ADAPTER = True
LABELS = ["NORMAL", "DROWSY", "LOOKING_AWAY", "PHONE_USAGE"]
IMAGE_SIZE = 512
MAX_EPOCHS = 8
EARLY_STOPPING_PATIENCE = 3
MACRO_F1_SAMPLES_PER_CLASS = 150
SEED = 42

# Mild, online augmentation for selected training classes only.
TARGETED_AUGMENTATION_PROBABILITY = 0.50
TARGETED_AUGMENTATION_LABELS = {"DROWSY", "LOOKING_AWAY"}
# 0 = uniform sampling; 1 = inverse-frequency sampling. Square-root is gentler.
SAMPLER_POWER = 0.50

LANGGRAPH_MODULE_PATH = HOME / "EdgeVLM_finetune" / "langgraph_pipeline.py"
if not LANGGRAPH_MODULE_PATH.is_file():
    raise FileNotFoundError(
        f"Production preprocessing module not found: {LANGGRAPH_MODULE_PATH}"
    )

spec = importlib.util.spec_from_file_location("langgraph_pipeline", LANGGRAPH_MODULE_PATH)
lgp = importlib.util.module_from_spec(spec)
sys.modules["langgraph_pipeline"] = lgp
spec.loader.exec_module(lgp)

print("LangGraph preprocessing:", lgp.__file__)
print("Full-frame size:", lgp.FULL_FRAME_SIZE)
print("Face-crop size:", lgp.FACE_CROP_SIZE)
print("Haar fallback enabled:", lgp.ENABLE_HAAR_FACE_FALLBACK)
print("Previous adapter:", PREVIOUS_ADAPTER_DIR)
print("Use previous adapter:", START_FROM_PREVIOUS_ADAPTER)
print("Output directory:", OUTPUT_ROOT)
for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    print(path, "exists:", path.is_file())
    if not path.is_file():
        raise FileNotFoundError(path)


## Experiment changes

Keeps the MediaPipe-guided face crop and dual-crop model input. Adds mild brightness/contrast augmentation for DROWSY and LOOKING_AWAY training samples only; uses gentler square-root sampling rather than stacking augmentation with strong inverse-frequency sampling; evaluates Macro-F1 on up to 150 validation images per class with patience 3; saves checkpoints every 100 steps and validates required files before resuming. Crop review shows original/full/face images and reports texture statistics as warnings only. Validation and test remain unaugmented; test remains reserved for final evaluation.

This is a separate fresh output directory. It can warm-start from the configured previous adapter but will not resume or overwrite the older MediaPipe experiment.


## 2. Read, validate, and deduplicate Phase 2 data


In [ ]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)
datasets = {"train": train_df, "validation": val_df, "test": test_df}
required_columns = {"frame_path", "label", "source_dataset", "subject"}


def resolve_image_path(value):
    path = Path(str(value).strip().replace("\\", "/"))
    return path if path.is_absolute() else HOME / path


for split_name, dataframe in datasets.items():
    missing_columns = required_columns - set(dataframe.columns)
    if missing_columns:
        raise ValueError(f"{split_name} missing columns: {sorted(missing_columns)}")
    dataframe["label"] = dataframe["label"].astype(str).str.strip().str.upper()
    unknown = sorted(set(dataframe["label"]) - set(LABELS))
    if unknown:
        raise ValueError(f"{split_name} contains unknown labels: {unknown}")
    dataframe["resolved_path"] = dataframe["frame_path"].map(
        lambda value: str(resolve_image_path(value))
    )
    missing = [p for p in dataframe["resolved_path"] if not Path(p).is_file()]
    print(f"\n{split_name}: {len(dataframe)} rows | missing images: {len(missing)}")
    print(dataframe["label"].value_counts().reindex(LABELS, fill_value=0))
    if missing:
        print("First missing path:", missing[0])

if any(not Path(p).is_file() for df in datasets.values() for p in df["resolved_path"]):
    raise FileNotFoundError("Fix missing image paths before training.")

for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    image_overlap = set(datasets[left]["resolved_path"]) & set(datasets[right]["resolved_path"])
    print(f"Exact image overlap {left}/{right}:", len(image_overlap))
    if image_overlap:
        raise ValueError(f"Image leakage detected between {left} and {right}")


def subject_keys(dataframe):
    return set(zip(dataframe["source_dataset"].astype(str), dataframe["subject"].astype(str)))


for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    overlap = subject_keys(datasets[left]) & subject_keys(datasets[right])
    print(f"Subject overlap {left}/{right}:", len(overlap))
    if overlap:
        raise ValueError(f"Subject leakage detected between {left} and {right}: {list(overlap)[:5]}")

before = len(train_df)
train_df = train_df.drop_duplicates(subset=["resolved_path"], keep="first").reset_index(drop=True)
datasets["train"] = train_df
print("\nTraining duplicates removed:", before - len(train_df))
print("Final training rows:", len(train_df))
print(train_df["label"].value_counts().reindex(LABELS, fill_value=0))


## 3. Verify dual-crop preprocessing on real images


In [ ]:
import cv2
import numpy as np
from PIL import Image


def create_dual_crops(image_path):
    """Shared dual-crop preprocessing for training and all evaluation inference."""
    path = str(image_path)
    raw_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if raw_bgr is None:
        raise FileNotFoundError(f"OpenCV could not read: {image_path}")
    enhanced_rgb, frame_quality_ok = lgp.enhance_frame(raw_bgr)
    full_crop = lgp.resize_full_frame(enhanced_rgb)
    face_bbox = FACE_BBOX_BY_PATH.get(path)
    if path not in BBOX_PROCESSED_PATHS:
        face_bbox = detect_live_face_bbox(enhanced_rgb)
        BBOX_PROCESSED_PATHS.add(path)
        if face_bbox is not None:
            FACE_BBOX_BY_PATH[path] = face_bbox
    landmarks = {"face_bbox": face_bbox} if face_bbox is not None else None
    face_crop = lgp.extract_face_crop(enhanced_rgb, landmarks=landmarks)
    face_detected = face_bbox is not None or lgp.detect_face_bbox_haar(enhanced_rgb) is not None
    return full_crop, face_crop, bool(frame_quality_ok), bool(face_detected)


def run_preprocessing_sanity_check():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    visual_rows = pd.concat([
        train_df[train_df["label"] == label].sample(n=2, random_state=SEED)
        for label in LABELS
    ]).reset_index(drop=True)
    fig, axes = plt.subplots(len(visual_rows), 3, figsize=(11, 3 * len(visual_rows)))
    for index, row in visual_rows.iterrows():
        raw_bgr = cv2.imread(row["resolved_path"], cv2.IMREAD_COLOR)
        if raw_bgr is None:
            raise FileNotFoundError(row["resolved_path"])
        original_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
        full_crop, face_crop, quality_ok, face_detected = create_dual_crops(row["resolved_path"])
        axes[index, 0].imshow(original_rgb)
        axes[index, 0].set_title(f"{row['label']} | original")
        axes[index, 1].imshow(full_crop)
        axes[index, 1].set_title("Full-scene crop")
        axes[index, 2].imshow(face_crop)
        axes[index, 2].set_title(f"MediaPipe face crop | detected={face_detected}")
        for axis in axes[index]:
            axis.axis("off")
    plt.tight_layout()
    sanity_path = OUTPUT_ROOT / "preprocessing_sanity_check.png"
    fig.suptitle("Review original frame and both model crops (visual review only)")
    fig.savefig(sanity_path, dpi=110)
    plt.close(fig)
    print("Saved visual check:", sanity_path)

    print("\nMediaPipe-guided face-crop availability on 100 training images per class:")
    for label in LABELS:
        subset = train_df[train_df["label"] == label]
        sample = subset.sample(n=min(100, len(subset)), random_state=SEED)
        detected = 0
        texture = []
        for image_path in sample["resolved_path"]:
            _, face_crop, _, _ = create_dual_crops(image_path)
            detected += int(str(image_path) in FACE_BBOX_BY_PATH)
            texture.append(float(np.std(cv2.cvtColor(face_crop, cv2.COLOR_RGB2GRAY))))
        print(f"{label}: MediaPipe box={100 * detected / len(sample):.1f}% | "
              f"face-crop grayscale std min/median/max="
              f"{np.min(texture):.1f}/{np.median(texture):.1f}/{np.max(texture):.1f}")
    print("Texture values are review signals only; no images are automatically discarded.")


## MediaPipe-guided face crops and face-mesh cues

MediaPipe Face Landmarker is run once per unique source image. Its face landmarks are converted into a pixel-space bounding box and cached, alongside the eye-blink, eye-gaze, and jaw-opening blendshape cues. During training, validation Macro-F1 checks, and test inference, `create_dual_crops()` uses that same cached MediaPipe box to make the face crop. If no face is detected, the shared LangGraph helper uses its configured fallback. The full-frame crop is unchanged.

The cue text and crop boxes are derived only from image pixels—not ground-truth labels. This keeps crop preparation consistent across the notebook's training and evaluation paths. For a live camera deployment, run the same Face Landmarker and crop function on each incoming frame; do not rely on the dataset-path cache.

The official MediaPipe task bundle is downloaded on the first run if it is not already cached. This experiment uses a fresh output directory and does not overwrite the prior experiment's checkpoint directory.


In [ ]:
# Install MediaPipe only if this kernel does not already have it.
import json
import subprocess
import urllib.request
from tqdm.auto import tqdm

try:
    import mediapipe as mp
except ImportError:
    print("Installing the official MediaPipe Python package (network required on first run)...", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mediapipe"])
    import mediapipe as mp

MEDIAPIPE_CACHE = PROJECT_ROOT / "phase2_256m_all_techniques_mediapipe_phone_prompt" / "mediapipe_cache"
MEDIAPIPE_CACHE.mkdir(parents=True, exist_ok=True)
FACE_LANDMARKER_TASK = HOME / ".cache" / "mediapipe" / "face_landmarker.task"
FACE_LANDMARKER_TASK.parent.mkdir(parents=True, exist_ok=True)
FACE_LANDMARKER_URL = (
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
    "face_landmarker/float16/latest/face_landmarker.task"
)
if not FACE_LANDMARKER_TASK.is_file():
    print("Downloading the official Face Landmarker task model...", flush=True)
    urllib.request.urlretrieve(FACE_LANDMARKER_URL, FACE_LANDMARKER_TASK)
print("MediaPipe:", mp.__version__)
print("Face Landmarker task:", FACE_LANDMARKER_TASK)

MEDIAPIPE_CUES_CSV = MEDIAPIPE_CACHE / "face_blendshape_cues.csv"
FACE_BOXES_CSV = MEDIAPIPE_CACHE / "face_landmark_boxes.csv"
if MEDIAPIPE_CUES_CSV.is_file():
    _cache_df = pd.read_csv(MEDIAPIPE_CUES_CSV)
    cue_cache = dict(zip(_cache_df["resolved_path"].astype(str), _cache_df["cue_text"].astype(str)))
    print("Loaded cached MediaPipe cues:", len(cue_cache))
else:
    cue_cache = {}
if FACE_BOXES_CSV.is_file():
    _box_df = pd.read_csv(FACE_BOXES_CSV)
    face_bbox_by_path = {}
    bbox_processed = set(_box_df["resolved_path"].astype(str))
    for _, _row in _box_df.iterrows():
        if pd.notna(_row["face_bbox_json"]) and str(_row["face_bbox_json"]).strip():
            face_bbox_by_path[str(_row["resolved_path"])] = tuple(json.loads(_row["face_bbox_json"]))
    print("Loaded cached MediaPipe crop boxes:", len(face_bbox_by_path), "| processed:", len(bbox_processed))
else:
    face_bbox_by_path, bbox_processed = {}, set()

all_paths = list(dict.fromkeys(
    path
    for split_df in (train_df, val_df, test_df)
    for path in split_df["resolved_path"].astype(str).tolist()
))
uncached_paths = [path for path in all_paths if path not in cue_cache or path not in bbox_processed]
print(f"Unique images: {len(all_paths)} | cues/landmark boxes still to extract: {len(uncached_paths)}")

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

_mp_options = mp_vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(FACE_LANDMARKER_TASK)),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=False,
)
_cue_names = [
    "eyeBlinkLeft", "eyeBlinkRight",
    "eyeLookDownLeft", "eyeLookDownRight",
    "eyeLookUpLeft", "eyeLookUpRight",
    "eyeLookInLeft", "eyeLookInRight",
    "eyeLookOutLeft", "eyeLookOutRight",
    "jawOpen",
]

with mp_vision.FaceLandmarker.create_from_options(_mp_options) as _landmarker:
    for _index, _path in enumerate(tqdm(uncached_paths, desc="Extracting MediaPipe face cues", mininterval=2), start=1):
        try:
            _bgr = cv2.imread(_path, cv2.IMREAD_COLOR)
            if _bgr is None:
                _cue_text = "unavailable (image could not be read)"
            else:
                _rgb, _ = lgp.enhance_frame(_bgr)
                _mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(_rgb))
                _result = _landmarker.detect(_mp_image)
                _face_landmarks = _result.face_landmarks[0] if _result.face_landmarks else None
                if _face_landmarks:
                    _height, _width = _rgb.shape[:2]
                    _xs = [point.x * _width for point in _face_landmarks]
                    _ys = [point.y * _height for point in _face_landmarks]
                    _bbox = [max(0, int(min(_xs))), max(0, int(min(_ys))),
                             min(_width, int(max(_xs)) + 1), min(_height, int(max(_ys)) + 1)]
                    if _bbox[2] > _bbox[0] and _bbox[3] > _bbox[1]:
                        face_bbox_by_path[_path] = tuple(_bbox)
                if not _result.face_blendshapes:
                    _cue_text = "unavailable (no face detected)"
                else:
                    _scores = {item.category_name: float(item.score) for item in _result.face_blendshapes[0]}
                    _cue_text = ", ".join(f"{name}={_scores.get(name, 0.0):.2f}" for name in _cue_names)
        except Exception as _error:
            _cue_text = f"unavailable (MediaPipe error: {type(_error).__name__})"
        cue_cache[_path] = _cue_text
        bbox_processed.add(_path)
        if _index % 250 == 0 or _index == len(uncached_paths):
            pd.DataFrame(
                [{"resolved_path": path, "cue_text": text} for path, text in cue_cache.items()]
            ).to_csv(MEDIAPIPE_CUES_CSV, index=False)
            pd.DataFrame([
                {"resolved_path": path, "face_bbox_json": json.dumps(face_bbox_by_path[path]) if path in face_bbox_by_path else ""}
                for path in sorted(bbox_processed)
            ]).to_csv(FACE_BOXES_CSV, index=False)
            print(f"Cached cues and landmark boxes for {len(bbox_processed)}/{len(all_paths)} images", flush=True)

# Ensure an initial cache exists even if every path was already cached.
pd.DataFrame(
    [{"resolved_path": path, "cue_text": cue_cache[path]} for path in all_paths]
).to_csv(MEDIAPIPE_CUES_CSV, index=False)
pd.DataFrame([
    {"resolved_path": path, "face_bbox_json": json.dumps(face_bbox_by_path[path]) if path in face_bbox_by_path else ""}
    for path in sorted(bbox_processed)
]).to_csv(FACE_BOXES_CSV, index=False)
CUE_BY_PATH = cue_cache
FACE_BBOX_BY_PATH = face_bbox_by_path
BBOX_PROCESSED_PATHS = bbox_processed
LIVE_FACE_LANDMARKER = mp_vision.FaceLandmarker.create_from_options(_mp_options)

def detect_live_face_bbox(rgb_image):
    """Run the same Face Landmarker on an unseen/live image frame."""
    _image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb_image))
    _detected = LIVE_FACE_LANDMARKER.detect(_image)
    if not _detected.face_landmarks:
        return None
    _h, _w = rgb_image.shape[:2]
    _points = _detected.face_landmarks[0]
    _xs = [point.x * _w for point in _points]
    _ys = [point.y * _h for point in _points]
    _box = (max(0, int(min(_xs))), max(0, int(min(_ys))), min(_w, int(max(_xs)) + 1), min(_h, int(max(_ys)) + 1))
    return _box if _box[2] > _box[0] and _box[3] > _box[1] else None
print("MediaPipe cue cache:", MEDIAPIPE_CUES_CSV)
print("MediaPipe face-box cache:", FACE_BOXES_CSV)
print("Images with MediaPipe face boxes:", len(FACE_BBOX_BY_PATH), "/", len(all_paths))
print("Unavailable cue rows:", sum(str(CUE_BY_PATH[p]).startswith("unavailable") for p in all_paths))
print("Example cue:", CUE_BY_PATH[all_paths[0]])
RUN_PREPROCESSING_SANITY_CHECK = True
if RUN_PREPROCESSING_SANITY_CHECK:
    run_preprocessing_sanity_check()
else:
    print("Preprocessing sanity check skipped.")


The `full_crop` is the complete **available input file**. If the dataset file is
already a face-only frame, this step cannot recreate missing hands or cabin context.
The saved sanity-check image must therefore be reviewed before trusting PHONE_USAGE
or LOOKING_AWAY performance.


## 4. Verify the existing offline training environment


In [ ]:
import torch
import transformers
import tokenizers
import peft
import accelerate
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel, LoraConfig, TaskType, get_peft_model

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook.")
print("Visible GPUs:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("OpenCV:", cv2.__version__)

if transformers.__version__ != "4.46.3":
    raise RuntimeError("Use Transformers 4.46.3 for this reproducible experiment.")
if tokenizers.__version__ != "0.20.3":
    raise RuntimeError("Use Tokenizers 0.20.3 for this reproducible experiment.")


## 5. Build a 256M starting model and attach vision+text LoRA


In [ ]:
import gc

device = torch.device("cuda:0")
dtype = torch.float16


def load_plain_base():
    options = {
        "cache_dir": str(MODEL_CACHE),
        "torch_dtype": dtype,
        "low_cpu_mem_usage": True,
    }
    try:
        return AutoModelForVision2Seq.from_pretrained(
            MODEL_ID, local_files_only=True, **options
        )
    except OSError:
        print("Local cache incomplete; attempting to download missing base-model files...")
        return AutoModelForVision2Seq.from_pretrained(MODEL_ID, **options)


def load_starting_base():
    base = load_plain_base()
    if START_FROM_PREVIOUS_ADAPTER:
        if PREVIOUS_ADAPTER_DIR is None:
            raise FileNotFoundError(
                "START_FROM_PREVIOUS_ADAPTER=True but the previous adapter was not found."
            )
        previous_model = PeftModel.from_pretrained(base, str(PREVIOUS_ADAPTER_DIR))
        base = previous_model.merge_and_unload()
        print("Merged previous good text-LoRA adapter into the 256M starting base.")
    else:
        print("Starting from the untouched pretrained 256M base.")
    return base


processor_source = (
    str(PREVIOUS_ADAPTER_DIR)
    if START_FROM_PREVIOUS_ADAPTER and PREVIOUS_ADAPTER_DIR is not None
    else MODEL_ID
)
processor = AutoProcessor.from_pretrained(
    processor_source,
    cache_dir=str(MODEL_CACHE),
    local_files_only=True,
)
model = load_starting_base()

ATTENTION_SUFFIXES = ("q_proj", "k_proj", "v_proj", "o_proj", "out_proj")
text_targets, vision_targets = [], []
for name, module in model.named_modules():
    if not isinstance(module, torch.nn.Linear):
        continue
    if not name.endswith(ATTENTION_SUFFIXES):
        continue
    if ".text_model." in name:
        text_targets.append(name)
    elif ".vision_model." in name:
        vision_targets.append(name)

print("Text attention targets:", len(text_targets))
print("Vision attention targets:", len(vision_targets))
if not text_targets or not vision_targets:
    raise RuntimeError(
        "Could not find both text and vision attention projections. "
        "Inspect model.named_modules() before training."
    )

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=sorted(text_targets + vision_targets),
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model = get_peft_model(model, lora_config).to(device)
model.train()
model.print_trainable_parameters()


## 6. Dual-crop dataset and multimodal collator


In [ ]:
from torch.utils.data import Dataset, DataLoader
import random
from PIL import ImageEnhance

INSTRUCTION = (
    "You are shown two images from the same in-cabin driver-camera frame. "
    "The first is the full available frame; the second is the face-focused crop. "
    "Use both images and the optional MediaPipe facial measurements below as "
    "additional visual evidence. The measurements are cues, not ground-truth labels "
    "or guaranteed diagnoses. Distinguish the two labels carefully: choose "
    "PHONE_USAGE when visible evidence shows the driver holding or interacting "
    "with a phone, even if the driver is also looking away. Choose LOOKING_AWAY "
    "when the driver's gaze or head is directed away from the road and there is "
    "no clear evidence of phone use. Do not infer phone use from gaze alone. Use "
    "the full frame for phone/hand context and the face crop for eye/head direction. "
    "Return exactly one label from NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE. "
    "Do not explain or add punctuation."
)


def instruction_with_cues(resolved_path):
    cue_text = CUE_BY_PATH.get(str(resolved_path), "unavailable (not cached)")
    return INSTRUCTION + "\n\nMediaPipe face cues (blendshape scores from 0 to 1): " + cue_text


class DriverDualCropDataset(Dataset):
    def __init__(self, dataframe, augment_training=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.augment_training = bool(augment_training)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        full_crop, face_crop, quality_ok, face_detected = create_dual_crops(row["resolved_path"])
        full_image = Image.fromarray(full_crop).convert("RGB")
        face_image = Image.fromarray(face_crop).convert("RGB")
        if (self.augment_training and row["label"] in TARGETED_AUGMENTATION_LABELS
                and random.random() < TARGETED_AUGMENTATION_PROBABILITY):
            brightness = random.uniform(0.90, 1.10)
            contrast = random.uniform(0.90, 1.10)
            full_image = ImageEnhance.Brightness(full_image).enhance(brightness)
            face_image = ImageEnhance.Brightness(face_image).enhance(brightness)
            full_image = ImageEnhance.Contrast(full_image).enhance(contrast)
            face_image = ImageEnhance.Contrast(face_image).enhance(contrast)
        messages = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "image"},
                {"type": "text", "text": instruction_with_cues(row["resolved_path"])},
            ],
        }]
        return {
            "full_image": full_image,
            "face_image": face_image,
            "messages": messages,
            "target": row["label"],
            "path": row["resolved_path"],
        }


class DualCropCollator:
    def __init__(self, processor, image_size):
        self.processor = processor
        self.image_size = image_size

    def __call__(self, features):
        images = [[item["full_image"], item["face_image"]] for item in features]
        prompt_texts, full_texts = [], []
        for item in features:
            prompt_messages = item["messages"]
            full_messages = prompt_messages + [{
                "role": "assistant",
                "content": [{"type": "text", "text": item["target"]}],
            }]
            prompt_texts.append(self.processor.apply_chat_template(
                prompt_messages, add_generation_prompt=True, tokenize=False
            ))
            full_texts.append(self.processor.apply_chat_template(
                full_messages, add_generation_prompt=False, tokenize=False
            ))

        kwargs = {
            "return_tensors": "pt",
            "padding": True,
            "truncation": True,
            "size": {"longest_edge": self.image_size},
            "max_image_size": {"longest_edge": self.image_size},
        }
        batch = self.processor(text=full_texts, images=images, **kwargs)
        prompt_batch = self.processor(text=prompt_texts, images=images, **kwargs)
        labels = batch["input_ids"].clone()
        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1)
        for row_index, prompt_length in enumerate(prompt_lengths):
            labels[row_index, : int(prompt_length)] = -100
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100
        batch["labels"] = labels
        return batch


train_dataset = DriverDualCropDataset(train_df, augment_training=True)
val_dataset = DriverDualCropDataset(val_df, augment_training=False)
test_dataset = DriverDualCropDataset(test_df, augment_training=False)
print("Targeted augmentation:", TARGETED_AUGMENTATION_LABELS, "p=", TARGETED_AUGMENTATION_PROBABILITY)
print("Validation/test augmentation: disabled")
collator = DualCropCollator(processor, IMAGE_SIZE)

sample_batch = collator([train_dataset[0]])
print("pixel_values:", tuple(sample_batch["pixel_values"].shape))
if sample_batch["pixel_values"].shape[1] != 2:
    raise RuntimeError("Dual-crop validation failed: expected two images per sample.")

# Backward-pass smoke test without an optimizer step, so it does not modify weights.
sample_batch = {
    key: value.to(device) if torch.is_tensor(value) else value
    for key, value in sample_batch.items()
}
smoke_output = model(**sample_batch)
smoke_output.loss.backward()
model.zero_grad(set_to_none=True)
print("Non-updating dual-crop smoke-test loss:", float(smoke_output.loss.detach().cpu()))
del sample_batch, smoke_output
gc.collect()
torch.cuda.empty_cache()


## 7. Generated Macro-F1 model selection and early stopping


In [ ]:
import json
import time
from sklearn.metrics import f1_score
from transformers import TrainerCallback


def parse_prediction(raw_text):
    normalized = str(raw_text).strip().upper()
    exact = normalized in LABELS
    cleaned = normalized.strip(" .,;:!?'\"")
    return (cleaned if cleaned in LABELS else "INVALID"), exact


def predict_rows_batched(model_ref, processor_ref, rows_df, batch_size=8):
    predictions = []
    old_padding = processor_ref.tokenizer.padding_side
    processor_ref.tokenizer.padding_side = "left"
    try:
        for start in range(0, len(rows_df), batch_size):
            rows = rows_df.iloc[start : start + batch_size]
            images, prompts = [], []
            for _, row in rows.iterrows():
                full_crop, face_crop, _, _ = create_dual_crops(row["resolved_path"])
                images.append([
                    Image.fromarray(full_crop).convert("RGB"),
                    Image.fromarray(face_crop).convert("RGB"),
                ])
                messages = [{
                    "role": "user",
                    "content": [
                        {"type": "image"}, {"type": "image"},
                        {"type": "text", "text": instruction_with_cues(row["resolved_path"])},
                    ],
                }]
                prompts.append(processor_ref.apply_chat_template(
                    messages, add_generation_prompt=True, tokenize=False
                ))

            inputs = processor_ref(
                text=prompts,
                images=images,
                return_tensors="pt",
                padding=True,
                size={"longest_edge": IMAGE_SIZE},
                max_image_size={"longest_edge": IMAGE_SIZE},
            )
            inputs = {
                key: value.to(next(model_ref.parameters()).device)
                if torch.is_tensor(value) else value
                for key, value in inputs.items()
            }
            with torch.inference_mode():
                generated = model_ref.generate(
                    **inputs, max_new_tokens=12, do_sample=False
                )
            input_length = inputs["input_ids"].shape[-1]
            raw_outputs = processor_ref.batch_decode(
                generated[:, input_length:], skip_special_tokens=True
            )
            predictions.extend(parse_prediction(text)[0] for text in raw_outputs)
    finally:
        processor_ref.tokenizer.padding_side = old_padding
    return predictions


macro_rows = []
for label in LABELS:
    class_rows = val_df[val_df["label"] == label]
    count = min(MACRO_F1_SAMPLES_PER_CLASS, len(class_rows))
    macro_rows.append(class_rows.sample(n=count, random_state=SEED))
macro_f1_df = pd.concat(macro_rows).reset_index(drop=True)
print("Macro-F1 validation subset:", macro_f1_df["label"].value_counts().to_dict())


class MacroF1EarlyStopping(TrainerCallback):
    def __init__(self, processor_ref, check_df, best_dir, patience):
        self.processor_ref = processor_ref
        self.check_df = check_df
        self.best_dir = Path(best_dir)
        self.patience = int(patience)
        self.log_path = OUTPUT_ROOT / "macro_f1_epoch_log.json"
        self.history = []
        self.best_macro_f1 = -1.0
        self.epochs_without_improvement = 0
        if self.log_path.is_file():
            self.history = json.loads(self.log_path.read_text(encoding="utf-8"))
            if self.history:
                scores = [float(row["macro_f1"]) for row in self.history]
                self.best_macro_f1 = max(scores)
                best_index = max(range(len(scores)), key=scores.__getitem__)
                self.epochs_without_improvement = len(scores) - 1 - best_index

    def on_epoch_end(self, args, state, control, **kwargs):
        model_ref = kwargs.get("model")
        if model_ref is None:
            return control
        model_ref.eval()
        started = time.perf_counter()
        try:
            predicted = predict_rows_batched(
                model_ref, self.processor_ref, self.check_df, batch_size=8
            )
            true = self.check_df["label"].astype(str).tolist()
            macro_f1 = f1_score(
                true, predicted, labels=LABELS, average="macro", zero_division=0
            )
            per_class = f1_score(
                true, predicted, labels=LABELS, average=None, zero_division=0
            )
            improved = macro_f1 > self.best_macro_f1
            if improved:
                self.best_macro_f1 = float(macro_f1)
                self.epochs_without_improvement = 0
                self.best_dir.mkdir(parents=True, exist_ok=True)
                model_ref.save_pretrained(str(self.best_dir))
                self.processor_ref.save_pretrained(str(self.best_dir))
            else:
                self.epochs_without_improvement += 1

            record = {
                "epoch": float(state.epoch),
                "macro_f1": float(macro_f1),
                "per_class_f1": dict(zip(LABELS, map(float, per_class))),
                "improved": bool(improved),
                "seconds": float(time.perf_counter() - started),
            }
            self.history.append(record)
            self.log_path.write_text(json.dumps(self.history, indent=2), encoding="utf-8")
            print(
                f"Macro-F1 epoch {state.epoch:.2f}: {macro_f1:.4f} | "
                f"best {self.best_macro_f1:.4f} | per class "
                f"{dict(zip(LABELS, np.round(per_class, 3)))}",
                flush=True,
            )
            if self.epochs_without_improvement >= self.patience:
                print("Macro-F1 patience exhausted; stopping training.", flush=True)
                control.should_training_stop = True
        finally:
            model_ref.train()
        return control


## 8. Terminal-friendly training ETA


In [ ]:
from datetime import datetime, timedelta

PROGRESS_FILE = OUTPUT_ROOT / "training_progress.txt"
PROGRESS_HISTORY = OUTPUT_ROOT / "training_progress_history.log"


def format_duration(seconds):
    hours, remainder = divmod(max(0, int(seconds)), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


class ETAProgressCallback(TrainerCallback):
    def __init__(self, update_every=25):
        self.update_every = int(update_every)
        self.start_time = None
        self.start_step = 0

    def _write(self, message):
        PROGRESS_FILE.write_text(message + "\n", encoding="utf-8")
        with PROGRESS_HISTORY.open("a", encoding="utf-8") as handle:
            handle.write(message + "\n")
        print(message, flush=True)

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        self.start_step = int(state.global_step)
        self._write(
            f"Training started | 0.00% | step {state.global_step}/{state.max_steps} | "
            f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        )

    def on_step_end(self, args, state, control, **kwargs):
        step = int(state.global_step)
        if step != int(state.max_steps) and step % self.update_every:
            return
        elapsed = time.time() - self.start_time
        completed = max(1, step - self.start_step)
        seconds_per_step = elapsed / completed
        remaining = max(0, int(state.max_steps) - step) * seconds_per_step
        percentage = 100 * step / max(1, int(state.max_steps))
        finish = datetime.now() + timedelta(seconds=remaining)
        self._write(
            f"{percentage:6.2f}% | step {step}/{state.max_steps} | "
            f"elapsed {format_duration(elapsed)} | estimated remaining "
            f"{format_duration(remaining)} | ETA {finish.strftime('%Y-%m-%d %H:%M:%S')}"
        )


## 9. Balanced multi-epoch fine-tuning


In [ ]:
from collections import Counter
import json
from torch.utils.data import WeightedRandomSampler
from transformers import TrainingArguments, Trainer, set_seed

set_seed(SEED)

class BalancedTrainer(Trainer):
    def _get_train_sampler(self):
        labels = self.train_dataset.dataframe["label"].astype(str).tolist()
        counts = Counter(labels)
        weights = torch.tensor(
            [1.0 / (counts[label] ** SAMPLER_POWER) for label in labels], dtype=torch.double
        )
        return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

def is_complete_checkpoint(path):
    required = ["trainer_state.json", "optimizer.pt", "scheduler.pt", "rng_state.pth"]
    if not all((path / name).is_file() for name in required):
        return False
    if not any((path / name).is_file() for name in
               ("adapter_model.safetensors", "model.safetensors", "pytorch_model.bin")):
        return False
    try:
        state = json.loads((path / "trainer_state.json").read_text(encoding="utf-8"))
        return int(state.get("global_step", 0)) > 0
    except (OSError, ValueError, TypeError):
        return False

checkpoint_candidates = sorted(
    (p for p in CHECKPOINT_DIR.glob("checkpoint-*") if p.is_dir()),
    key=lambda p: int(p.name.rsplit("-", 1)[-1]) if p.name.rsplit("-", 1)[-1].isdigit() else -1,
    reverse=True,
)
last_checkpoint = next((p for p in checkpoint_candidates if is_complete_checkpoint(p)), None)
if checkpoint_candidates and last_checkpoint is None:
    print("Checkpoint folders exist, but none passed resume checks; starting fresh.")

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR), num_train_epochs=MAX_EPOCHS,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, learning_rate=1e-4, warmup_ratio=0.05,
    lr_scheduler_type="cosine", weight_decay=0.01, fp16=True, logging_steps=50,
    disable_tqdm=True, eval_strategy="epoch", save_strategy="steps", save_steps=100,
    save_total_limit=3, load_best_model_at_end=False, remove_unused_columns=False,
    dataloader_num_workers=0, optim="adamw_torch", report_to="none", seed=SEED,
)

macro_callback = MacroF1EarlyStopping(
    processor_ref=processor, check_df=macro_f1_df,
    best_dir=BEST_ADAPTER_DIR, patience=EARLY_STOPPING_PATIENCE,
)
trainer = BalancedTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=collator,
    callbacks=[ETAProgressCallback(update_every=25), macro_callback],
)

print("Resume checkpoint:", last_checkpoint)
print("Sampling: square-root inverse-frequency (gentler rebalancing)")
print("Saving complete checkpoints every", training_args.save_steps, "optimizer steps")
print("Starting dual-crop vision+text LoRA training...")
print("Progress file:", PROGRESS_FILE)
train_result = trainer.train(resume_from_checkpoint=str(last_checkpoint) if last_checkpoint else None)
print("Training ended:", train_result)
trainer.save_model(str(LAST_ADAPTER_DIR))
processor.save_pretrained(str(LAST_ADAPTER_DIR))
print("Last adapter:", LAST_ADAPTER_DIR)
print("Macro-F1-best adapter:", BEST_ADAPTER_DIR)


## 10. Reload the Macro-F1-best model


In [ ]:
if not (BEST_ADAPTER_DIR / "adapter_model.safetensors").is_file():
    raise FileNotFoundError("Macro-F1-best adapter was not saved.")

del trainer, model
gc.collect()
torch.cuda.empty_cache()

eval_processor = AutoProcessor.from_pretrained(str(BEST_ADAPTER_DIR), local_files_only=True)
eval_base = load_starting_base()
eval_model = PeftModel.from_pretrained(eval_base, str(BEST_ADAPTER_DIR)).to(device)
eval_model.eval()
eval_model.config.use_cache = True
print("Loaded Macro-F1-best model:", BEST_ADAPTER_DIR)


## 11. Full, resumable Phase 2 test evaluation


In [ ]:
from tqdm.auto import tqdm

PREDICTIONS_CSV = OUTPUT_ROOT / "phase2_256m_all_techniques_mediapipe_test_predictions.csv"
SAVE_EVERY = 250


def predict_test_row(row):
    full_crop, face_crop, _, _ = create_dual_crops(row["resolved_path"])
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"}, {"type": "image"},
            {"type": "text", "text": instruction_with_cues(row["resolved_path"])},
        ],
    }]
    prompt = eval_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = eval_processor(
        text=prompt,
        images=[[Image.fromarray(full_crop), Image.fromarray(face_crop)]],
        return_tensors="pt",
        size={"longest_edge": IMAGE_SIZE},
        max_image_size={"longest_edge": IMAGE_SIZE},
    )
    inputs = {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in inputs.items()
    }
    started = time.perf_counter()
    with torch.inference_mode():
        generated = eval_model.generate(
            **inputs, max_new_tokens=12, do_sample=False
        )
    latency_ms = (time.perf_counter() - started) * 1000
    input_length = inputs["input_ids"].shape[-1]
    raw = eval_processor.batch_decode(
        generated[:, input_length:], skip_special_tokens=True
    )[0].strip()
    prediction, exact = parse_prediction(raw)
    return raw, prediction, exact, latency_ms


if PREDICTIONS_CSV.is_file():
    records = pd.read_csv(PREDICTIONS_CSV).to_dict("records")
    finished = {int(row["row_index"]) for row in records}
    print("Resuming after", len(finished), "test rows.")
else:
    records, finished = [], set()

torch.cuda.reset_peak_memory_stats()
remaining = [index for index in range(len(test_df)) if index not in finished]
for completed, index in enumerate(
    tqdm(remaining, desc="Evaluating full Phase 2 test set", mininterval=2), start=1
):
    row = test_df.iloc[index]
    raw, prediction, exact, latency_ms = predict_test_row(row)
    records.append({
        "row_index": index,
        "frame_path": row["frame_path"],
        "true_label": row["label"],
        "raw_output": raw,
        "prediction": prediction,
        "exact_label_valid": exact,
        "latency_ms": latency_ms,
    })
    if completed % SAVE_EVERY == 0:
        pd.DataFrame(records).sort_values("row_index").to_csv(PREDICTIONS_CSV, index=False)

predictions_df = (
    pd.DataFrame(records)
    .sort_values("row_index")
    .drop_duplicates("row_index", keep="last")
)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)
print("Saved test rows:", len(predictions_df), "/", len(test_df))
print("Predictions:", PREDICTIONS_CSV)


## 12. Final metrics


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

if len(predictions_df) != len(test_df):
    raise RuntimeError("Full test evaluation is incomplete.")

y_true = predictions_df["true_label"].astype(str).to_numpy()
y_pred = predictions_df["prediction"].astype(str).to_numpy()
latencies = predictions_df["latency_ms"].astype(float).to_numpy()

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=LABELS, zero_division=0
)
unsafe_true = np.isin(y_true, ["DROWSY", "LOOKING_AWAY", "PHONE_USAGE"])
normal_true = y_true == "NORMAL"
unsafe_pred = np.isin(y_pred, ["DROWSY", "LOOKING_AWAY", "PHONE_USAGE"])

unsafe_to_safe_miss = np.mean(np.isin(y_pred[unsafe_true], ["NORMAL", "INVALID"]))
unsafe_class_error = np.mean(y_pred[unsafe_true] != y_true[unsafe_true])
false_alarm = np.mean(unsafe_pred[normal_true])
exact_validity = predictions_df["exact_label_valid"].astype(bool).mean()
parser_failure = np.mean(y_pred == "INVALID")
matrix_labels = LABELS + ["INVALID"]
cm = confusion_matrix(y_true, y_pred, labels=matrix_labels)

print("\n========== 256M ALL-TECHNIQUES PHASE 2 TEST ==========")
print("Images:", len(predictions_df))
print("Accuracy:", round(accuracy_score(y_true, y_pred) * 100, 2), "%")
print("Balanced accuracy:", round(recall.mean() * 100, 2), "%")
print("Macro-F1:", round(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0) * 100, 2), "%")
print("Exact-label validity:", round(exact_validity * 100, 2), "%")
print("Parser failure:", round(parser_failure * 100, 2), "%")
print("\nPer-class results:")
for label, p, r, score, count in zip(LABELS, precision, recall, f1, support):
    print(f"{label}: precision={p:.3f}, recall={r:.3f}, F1={score:.3f}, samples={count}")
print("\nConfusion matrix:")
print(pd.DataFrame(cm, index=matrix_labels, columns=matrix_labels))
print("Unsafe-to-safe miss rate:", round(unsafe_to_safe_miss * 100, 2), "%")
print("Unsafe-class error rate:", round(unsafe_class_error * 100, 2), "%")
print("False-alarm rate:", round(false_alarm * 100, 2), "%")
print("Mean latency (ms):", round(float(np.mean(latencies)), 2))
print("P95 latency (ms):", round(float(np.percentile(latencies, 95)), 2))
print("Peak GPU memory (MB):", round(torch.cuda.max_memory_allocated() / 1024**2, 2))


## 13. Optional merged FP32 deployment export


In [ ]:
CREATE_MERGED_FP32 = False

if CREATE_MERGED_FP32:
    MERGED_DIR = OUTPUT_ROOT / "merged_model_256m_all_techniques_fp32"
    merged_model = eval_model.merge_and_unload().float()
    merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
    eval_processor.save_pretrained(str(MERGED_DIR))
    print("Merged FP32 model:", MERGED_DIR)
else:
    print("Merge skipped until the new test results are accepted.")
